# Summoning daemons

The daemon runs services side by side until one ends or it is stopped, when the
others are stopped too. A service is anything with `run` and `stop`.

In [1]:
import asyncio

from amoorion.daemon import Daemon, Service


class Ticker(Service):
    name = "ticker"

    def __init__(
        self,
        times,
    ):

        super().__init__()

        self.times, self.ticks, self.stopped = times, 0, False

    async def run(self):

        while not self.stopped and (
            self.times is None or self.ticks < self.times
        ):
            self.ticks += 1

            await asyncio.sleep(0.01)

    async def stop(self):

        self.stopped = True


short, endless = Ticker(3), Ticker(None)


async def summon():

    await Daemon(
        [
            short,
            endless,
        ]
    ).run()


await summon()

short.ticks, endless.stopped

(3, True)

A service that fails brings the daemon down with its error, so a supervisor sees it:

In [2]:
class Broken(Service):
    name = "broken"

    async def run(self):

        raise RuntimeError("no connection")


async def summon_broken():

    try:
        await Daemon(
            [
                Broken(),
                Ticker(None),
            ]
        ).run()
    except RuntimeError as error:
        return f"daemon stopped: {error}"


await summon_broken()

'daemon stopped: no connection'

For real, the API and hooks, and NATS when `.env` names a server:

```sh
uv run amoorion daemon --host 127.0.0.1 --port 8000
```